# APIM Multi-Region Failover Lab

> **Prove your API gateway survives a region failure — with real Azure infrastructure.**

This notebook deploys Azure API Management **Premium** across two regions (East US + West US 2),
tests health on both regional gateways, and simulates DR scenarios using `disableGateway`.

![Architecture Diagram](../../docs/images/jpny-architecture.jpeg)

**Estimated time:** ~45 min (APIM Premium provisioning takes 30-40 min)

**Cost warning:** APIM Premium ≈ $2.80/hr per unit (2 units total). Run cleanup when done.

## 0️⃣ Initialize — Variables & Dependencies

In [ ]:
import os
import sys
import json
import time
import subprocess
from datetime import datetime, timezone

# Install dependencies if needed
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
import requests

# Add shared utilities to path
sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "shared"))
from utils import run_az_cli, check_region_health, check_default_gateway, toggle_gateway, send_traffic_burst

# ── Configuration ────────────────────────────────────────────────────────
SUFFIX          = datetime.now(timezone.utc).strftime("%m%d%H%M")
BASE_NAME       = f"apimmr-{SUFFIX}"           # Base name for all resources
RESOURCE_GROUP  = f"{BASE_NAME}-rg"             # Resource group name
PRIMARY_REGION  = "eastus"                      # Primary region
SECONDARY_REGION = "westus2"                    # Secondary region
PUBLISHER_EMAIL = "admin@contoso.com"           # ← Change to your email

# These will be populated after deployment
APIM_NAME               = None
APIM_GATEWAY_URL        = None
APIM_PRIMARY_REGIONAL   = None
APIM_SECONDARY_REGIONAL = None

print(f"Base name:        {BASE_NAME}")
print(f"Resource group:   {RESOURCE_GROUP}")
print(f"Primary region:   {PRIMARY_REGION}")
print(f"Secondary region: {SECONDARY_REGION}")
print(f"Publisher email:  {PUBLISHER_EMAIL}")

## 1️⃣ Verify Azure CLI

In [ ]:
# Verify Azure CLI is installed and logged in
account = run_az_cli("account show")
print(f"✅ Logged in as:    {account['user']['name']}")
print(f"   Subscription:   {account['name']}")
print(f"   Subscription ID:{account['id']}")
print(f"   Tenant ID:      {account['tenantId']}")

## 2️⃣ Deploy Infrastructure

This cell deploys the full stack via Bicep:
- Log Analytics + Application Insights
- Container Apps in East US and West US 2
- APIM Premium with both regions

> ⏱️ **This takes ~30-45 minutes** (APIM Premium provisioning).

In [ ]:
# Create resource group
print(f"Creating resource group '{RESOURCE_GROUP}' in {PRIMARY_REGION}...")
run_az_cli(
    f'group create --name {RESOURCE_GROUP} --location {PRIMARY_REGION}',
    parse_json=False
)
print("✅ Resource group created.")

# Deploy Bicep template
BICEP_PATH = os.path.join(os.getcwd(), "main.bicep")
print(f"\nDeploying Bicep template: {BICEP_PATH}")
print("⏳ This will take ~30-45 minutes for APIM Premium provisioning...")
print(f"   Start time: {datetime.now(timezone.utc).strftime('%H:%M:%S UTC')}")

deployment = run_az_cli(
    f'deployment group create '
    f'--resource-group {RESOURCE_GROUP} '
    f'--template-file "{BICEP_PATH}" '
    f'--parameters baseName={BASE_NAME} '
    f'  publisherEmail={PUBLISHER_EMAIL} '
    f'  primaryLocation={PRIMARY_REGION} '
    f'  secondaryLocation={SECONDARY_REGION} '
    f'--verbose'
)

print(f"\n✅ Deployment completed!")
print(f"   End time: {datetime.now(timezone.utc).strftime('%H:%M:%S UTC')}")
print(f"   State: {deployment['properties']['provisioningState']}")

## 3️⃣ Retrieve Deployment Outputs

In [ ]:

# Get outputs from the deployment
outputs = deployment['properties']['outputs']

APIM_NAME               = outputs['apimName']['value']
APIM_GATEWAY_URL        = outputs['apimGatewayUrl']['value']
APIM_PRIMARY_REGIONAL   = outputs['apimPrimaryRegionalUrl']['value']
APIM_SECONDARY_REGIONAL = outputs['apimSecondaryRegionalUrl']['value']
PRIMARY_CONTAINER_URL   = outputs['primaryContainerAppUrl']['value']
SECONDARY_CONTAINER_URL = outputs['secondaryContainerAppUrl']['value']

print("═" * 70)
print("                    DEPLOYMENT OUTPUTS")
print("═" * 70)
print(f"  APIM Name:              {APIM_NAME}")
print(f"  Default Gateway:        {APIM_GATEWAY_URL}")
print(f"  Primary Regional URL:   {APIM_PRIMARY_REGIONAL}")
print(f"  Secondary Regional URL: {APIM_SECONDARY_REGIONAL}")
print(f"  Primary Backend:        {PRIMARY_CONTAINER_URL}")
print(f"  Secondary Backend:      {SECONDARY_CONTAINER_URL}")
print(f"  Auth:                   None (subscriptionRequired=false)")
print("═" * 70)


## 4️⃣ Test Default Gateway

Send requests to the load-balanced default gateway URL.
APIM routes to the nearest regional gateway (East US or West US 2).

In [ ]:
print("Testing default gateway...")
print(f"URL: {APIM_GATEWAY_URL}/multiregion-health-api/health\n")

for i in range(5):
    result = check_default_gateway(APIM_GATEWAY_URL)
    status_icon = "✅" if result['status'] == 'healthy' else "❌"
    print(
        f"  Request {i+1}: {status_icon} "
        f"region={result['region']:10s} "
        f"latency={result['latency_ms']:>7.1f}ms "
        f"status={result['status_code']}"
    )
    time.sleep(0.5)

print("\n✅ Default gateway is responding.")

## 5️⃣ Test Regional Endpoints Directly

Hit each regional gateway URL to verify both regions are active.

In [ ]:
print("Testing regional endpoints directly...\n")

# Primary region
print(f"─── Primary: East US ───")
print(f"URL: {APIM_PRIMARY_REGIONAL}")
primary_health = check_region_health(APIM_PRIMARY_REGIONAL)
status_icon = "✅" if primary_health['status'] == 'healthy' else "❌"
print(f"  {status_icon} Status: {primary_health['status']}")
print(f"     Region: {primary_health['region']}")
print(f"     Latency: {primary_health['latency_ms']}ms")
if primary_health.get('body'):
    print(f"     Body: {json.dumps(primary_health['body'], indent=2)}")

print()

# Secondary region
print(f"─── Secondary: West US 2 ───")
print(f"URL: {APIM_SECONDARY_REGIONAL}")
secondary_health = check_region_health(APIM_SECONDARY_REGIONAL)
status_icon = "✅" if secondary_health['status'] == 'healthy' else "❌"
print(f"  {status_icon} Status: {secondary_health['status']}")
print(f"     Region: {secondary_health['region']}")
print(f"     Latency: {secondary_health['latency_ms']}ms")
if secondary_health.get('body'):
    print(f"     Body: {json.dumps(secondary_health['body'], indent=2)}")

print("\n✅ Both regional gateways are active.")

## 6️⃣ Region Health Summary

Display a summary table of both regions' health status.

In [ ]:
def print_health_table():
    """Check and display health for all regions."""
    regions = [
        ("East US (Primary)",   APIM_PRIMARY_REGIONAL),
        ("West US 2 (Secondary)", APIM_SECONDARY_REGIONAL),
    ]

    print("\n" + "═" * 80)
    print(f"  {'Region':<28s} {'Status':<14s} {'Backend Region':<16s} {'Latency':>10s}")
    print("─" * 80)

    for name, url in regions:
        h = check_region_health(url)
        icon = "🟢" if h['status'] == 'healthy' else "🔴" if h['status'] == 'unhealthy' else "⚫"
        print(
            f"  {icon} {name:<26s} {h['status']:<14s} {h['region']:<16s} {h['latency_ms']:>8.1f}ms"
        )
        if h.get('error'):
            print(f"     Error: {h['error']}")

    print("═" * 80 + "\n")

print_health_table()

## 7️⃣ Simulate Failure — Disable Secondary Region (West US 2)

Uses `disableGateway=true` on the West US 2 additional location.
After this, **all traffic should be served by East US only**.

> This is the official Azure mechanism for DR drills.

In [ ]:
print("🔴 Disabling West US 2 (secondary) gateway...")
result = toggle_gateway(
    apim_name=APIM_NAME,
    resource_group=RESOURCE_GROUP,
    location="West US 2",
    disable=True,
    is_primary=False,
)
print(f"   {result}")

print("\n⏳ Waiting 60 seconds for gateway changes to propagate...")
time.sleep(60)

# Verify: send 10 requests to default gateway — should all go to East US
print("\n📊 Sending 10 requests to default gateway...")
results = send_traffic_burst(APIM_GATEWAY_URL, num_requests=10)
for r in results:
    print(f"  Request {r['request_num']:>2d}: region={r['region']:<10s} latency={r['latency_ms']:>7.1f}ms")

region_counts = {}
for r in results:
    region_counts[r['region']] = region_counts.get(r['region'], 0) + 1

print(f"\n📈 Traffic distribution: {region_counts}")
if 'westus2' not in region_counts:
    print("✅ SUCCESS: No traffic reached West US 2 — failover confirmed!")
else:
    print("⚠️  Some traffic still reached West US 2 (propagation may still be in progress).")

# Show health table
print_health_table()

## 8️⃣ Simulate Failure — Disable Primary Region (East US)

Re-enable West US 2, then disable East US using the REST API.
After this, **all traffic should fail over to West US 2**.

In [ ]:
# Step 1: Re-enable secondary
print("🟢 Re-enabling West US 2 (secondary) gateway...")
result = toggle_gateway(
    apim_name=APIM_NAME,
    resource_group=RESOURCE_GROUP,
    location="West US 2",
    disable=False,
    is_primary=False,
)
print(f"   {result}")

print("\n⏳ Waiting 60 seconds for secondary to come back online...")
time.sleep(60)

# Step 2: Disable primary
print("\n🔴 Disabling East US (primary) gateway...")
result = toggle_gateway(
    apim_name=APIM_NAME,
    resource_group=RESOURCE_GROUP,
    location="East US",
    disable=True,
    is_primary=True,
)
print(f"   {result}")

print("\n⏳ Waiting 90 seconds for gateway changes to propagate...")
time.sleep(90)

# Verify: send 10 requests — should all go to West US 2
print("\n📊 Sending 10 requests to default gateway...")
results = send_traffic_burst(APIM_GATEWAY_URL, num_requests=10)
for r in results:
    print(f"  Request {r['request_num']:>2d}: region={r['region']:<10s} latency={r['latency_ms']:>7.1f}ms")

region_counts = {}
for r in results:
    region_counts[r['region']] = region_counts.get(r['region'], 0) + 1

print(f"\n📈 Traffic distribution: {region_counts}")
if 'eastus' not in region_counts:
    print("✅ SUCCESS: No traffic reached East US — failover to West US 2 confirmed!")
else:
    print("⚠️  Some traffic still reached East US (propagation may still be in progress).")

# Show health table
print_health_table()

## 9️⃣ Restore All Gateways

Re-enable both regional gateways and verify normal operation.

In [ ]:
# Re-enable primary
print("🟢 Re-enabling East US (primary) gateway...")
result = toggle_gateway(
    apim_name=APIM_NAME,
    resource_group=RESOURCE_GROUP,
    location="East US",
    disable=False,
    is_primary=True,
)
print(f"   {result}")

print("\n⏳ Waiting 90 seconds for primary to come back online before updating secondary...")
time.sleep(90)

# Make sure secondary is also enabled
print("🟢 Ensuring West US 2 (secondary) gateway is enabled...")
result = toggle_gateway(
    apim_name=APIM_NAME,
    resource_group=RESOURCE_GROUP,
    location="West US 2",
    disable=False,
    is_primary=False,
)
print(f"   {result}")

print("\n⏳ Waiting 90 seconds for all gateways to come online...")
time.sleep(90)

# Verify both regions are back
print("\n📊 Sending 20 requests to default gateway...")
results = send_traffic_burst(APIM_GATEWAY_URL, num_requests=20)

region_counts = {}
for r in results:
    region_counts[r['region']] = region_counts.get(r['region'], 0) + 1

print(f"\n📈 Traffic distribution: {region_counts}")
if len(region_counts) >= 2:
    print("✅ SUCCESS: Traffic is flowing to both regions — full recovery confirmed!")
else:
    print(f"⚠️  Traffic only reached {list(region_counts.keys())} — may need more time to propagate.")

# Final health check
print_health_table()

## 🔟 Launch Streamlit Dashboard

Start the live monitoring dashboard for a visual view of region health.

> The dashboard runs in a separate process. Visit [http://localhost:8501](http://localhost:8501) in your browser.

In [ ]:
import subprocess

# Write a .env file for the dashboard
dashboard_dir = os.path.join(os.getcwd(), "..", "..", "dashboard")
env_file = os.path.join(dashboard_dir, ".env")

with open(env_file, "w") as f:
    f.write(f"APIM_NAME={APIM_NAME}\n")
    f.write(f"RESOURCE_GROUP={RESOURCE_GROUP}\n")
    f.write(f"APIM_GATEWAY_URL={APIM_GATEWAY_URL}\n")
    f.write(f"APIM_PRIMARY_REGIONAL_URL={APIM_PRIMARY_REGIONAL}\n")
    f.write(f"APIM_SECONDARY_REGIONAL_URL={APIM_SECONDARY_REGIONAL}\n")
    f.write(f"PRIMARY_REGION=East US\n")
    f.write(f"SECONDARY_REGION=West US 2\n")

print(f"✅ Dashboard .env written to {env_file}")
print("\n🚀 Starting Streamlit dashboard...")
print("   Open http://localhost:8501 in your browser")
print("   Press Ctrl+C in the terminal to stop\n")

dashboard_app = os.path.join(dashboard_dir, "app.py")
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", dashboard_app, "--server.headless", "true"],
    cwd=dashboard_dir,
)
print(f"Dashboard PID: {proc.pid}")

## 🗑️ Clean Up

Delete the resource group and all resources created during this lab.

> ⚠️ **This is irreversible.** Make sure you're done with the demo.

In [ ]:
# Stop dashboard if running
try:
    proc.terminate()
    print("Stopped Streamlit dashboard.")
except:
    pass

# Remove .env file
try:
    os.remove(env_file)
    print("Removed dashboard .env file.")
except:
    pass

print(f"\n🗑️  Deleting resource group '{RESOURCE_GROUP}'...")
print("   This may take a few minutes.\n")

run_az_cli(
    f'group delete --name {RESOURCE_GROUP} --yes --no-wait',
    parse_json=False,
)

print(f"✅ Resource group '{RESOURCE_GROUP}' deletion initiated (--no-wait).")
print("   Resources will be removed in the background.")
print("\n🏁 Lab complete!")